# 01 - Fundamentos de Isolation Forest y Árboles de Aislamiento (iTrees)

## Curso de Machine Learning: Modelos No Supervisados

En todos los algoritmos vistos hasta ahora (K-Means, DBSCAN, PCA), la detección de anomalías ocurría como un **subproducto secundario**: primero intentábamos modelar el perfil de las observaciones "normales" (mediante centroides o regiones de alta densidad) y luego definíamos como anomalía todo aquello que no encajara bien en el modelo normal.

Ese enfoque clásico sufre dos desventajas graves en sistemas de prevención de fraude de gran escala:
1. **Optimización para el objetivo equivocado**: El algoritmo gasta el $99\%$ de su esfuerzo computacional perfilando millones de transacciones legítimas en lugar de enfocarse directamente en aislar el fraude.
2. **Costo cuadrático de distancias**: Calcular distancias o densidades locales ($O(n^2)$) en tablas de millones de transacciones con decenas de variables es inviable en tiempo real.

**Isolation Forest** (*Bosque de Aislamiento*), propuesto por Fei Tony Liu, Kai Ming Ting y Zhi-Hua Zhou (2008), revolucionó la detección de anomalías al introducir un paradigma completamente nuevo: **en lugar de perfilar la normalidad, aísla explícitamente las anomalías mediante particiones aleatorias recursivas**, logrando una complejidad lineal $O(n \log \psi)$ y sin calcular jamás una sola distancia métrica.

---

### Contenidos
1. El Cambio de Paradigma: Perfilar la Normalidad vs. Aislar la Anomalía
2. Las Dos Propiedades Fundamentales de las Anomalías (*Few and Different*)
3. Anatomía de un Árbol de Aislamiento (*Isolation Tree - iTree*)
4. Intuición Geométrica: Por qué una Anomalía Requiere Menos Cortes que un Punto Normal
5. Formulación Matemática del Puntaje de Anomalía $s(x, \psi)$ y Factor Armónico $c(\psi)$
6. Invarianza de Escala: Por qué Isolation Forest NO Requiere `StandardScaler`
7. Implementación en Scikit-Learn (`IsolationForest`): `predict`, `decision_function` y `score_samples`
8. Visualización de Contornos de Aislamiento en un Caso de Fraude Transaccional
9. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Intuición Geométrica: ¿Qué es Aislar una Observación?

Las anomalías satisfacen dos condiciones universales: **son pocas (*few*)** y **tienen valores de atributos muy distintos a los normales (*different*)**.

Imaginemos que construimos un árbol binario completamente aleatorio (*Isolation Tree* o **iTree**) sobre una muestra de tamaño $\psi$ (por defecto $\psi = 256$):
1. Elegimos **al azar una variable** $j \in \{1, \dots, p\}$.
2. Elegimos **al azar un punto de corte uniforme** $t \sim \text{Uniform}(\min(x_j), \max(x_j))$.
3. Partimos el espacio en dos mitades ($x_j < t$ vs. $x_j \geq t$) y repetimos recursivamente hasta que cada punto quede completamente solo en una hoja terminal o se alcance la altura límite $\lceil \log_2(\psi) \rceil$.

### La Observación Clave:
- Un **punto normal** (ubicado en el centro denso de la nube de transacciones legítimas) requiere **muchísimas divisiones aleatorias** para quedar aislado de sus vecinos $\implies$ **Longitud de camino $h(x)$ GRANDE** (cae profundo en el árbol).
- Una **anomalía / fraude** (ubicada lejos en la periferia) queda separada del resto en los **primeros 1 o 2 cortes cerca de la raíz** $\implies$ **Longitud de camino $h(x)$ MUY CORTA**.


## 2. Formulación Matemática del Score de Anomalía

Sea $h(x)$ la longitud del camino (número de aristas desde el nodo raíz hasta la hoja donde queda aislado el punto $x$) en un árbol de aislamiento, y sea $E[h(x)]$ el promedio de $h(x)$ a través de los $T$ árboles del bosque.

Para normalizar $E[h(x)]$, Liu et al. utilizan el costo promedio de búsqueda sin éxito en un Árbol Binario de Búsqueda (*BST*) de $\psi$ elementos:
$$c(\psi) = 2 H(\psi - 1) - \frac{2(\psi - 1)}{\psi}, \quad \text{donde } H(k) \approx \ln(k) + \gamma_{\text{Euler}} \; (0.57721566)$$

El **Puntaje de Anomalía Original $s(x, \psi) \in (0, 1]$** se define como:
$$s(x, \psi) = 2^{-\frac{E[h(x)]}{c(\psi)}}$$

### Interpretación Matemática:
- **Si $E[h(x)] \to 0$ (Aislado inmediatamente cerca de la raíz)**: $s(x, \psi) \to 2^0 = \mathbf{1.0}$ $\implies$ **Anomalía segura / Fraude claro**.
- **Si $E[h(x)] = c(\psi)$ (Profundidad promedio)**: $s(x, \psi) = 2^{-1} = \mathbf{0.5}$ $\implies$ Comportamiento normal sin distinción.
- **Si $E[h(x)] \to \psi - 1$ (Muy profundo en el interior denso)**: $s(x, \psi) \to \mathbf{0.0}$ $\implies$ **Observación completamente normal (Inlier)**.

> **Nota sobre Scikit-Learn**: En `sklearn.ensemble.IsolationForest`, el método `.score_samples(X)` devuelve $-s(x, \psi)$ (valores más negativos indican mayor anomalía), y `.decision_function(X)` devuelve un puntaje desplazado donde **valores $< 0$ son predichos como anomalías (`-1`)** y **valores $\geq 0$ como normales (`+1`)**.


In [ ]:
# Demostración Visual: Construcción de la Superficie de Nivel de Isolation Forest en 2D
rng = np.random.default_rng(42)

#Dos grupos de transacciones normales (Compras diarias y Pagos de nomina/renta)
X_norm_1 = rng.normal(loc=[-2.0, -1.5], scale=[0.6, 0.6], size=(300, 2))
X_norm_2 = rng.normal(loc=[2.0, 2.0], scale=[0.8, 0.5], size=(300, 2))

# 30 Transacciones fraudulentas dispersas
X_fraude = rng.uniform(low=[-5.5, -5.0], high=[5.5, 5.5], size=(30, 2))
X_total = np.vstack([X_norm_1, X_norm_2, X_fraude])
y_real = np.array([1]*600 + [-1]*30)  # Convencion sklearn: +1 Normal, -1 Anomalia

# Entrenamiento de Isolation Forest
iforest = IsolationForest(n_estimators=150, max_samples=256, contamination=30/630, random_state=42)
y_pred = iforest.fit_predict(X_total)
scores_decision = iforest.decision_function(X_total)

# Malla para visualizar el campo escalar de longitudes de camino
xx, yy = np.meshgrid(np.linspace(-6, 6, 300), np.linspace(-5.5, 6, 300))
Z = iforest.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Panel 1: Mapa de calor de decision_function y frontera cero
cf = axes[0].contourf(xx, yy, Z, levels=25, cmap='RdYlBu', alpha=0.85)
axes[0].contour(xx, yy, Z, levels=[0.0], linewidths=2.5, colors='black', linestyles='--')
axes[0].scatter(X_total[y_real==1, 0], X_total[y_real==1, 1], c='white', s=25, edgecolors='k', alpha=0.8, label='Transacciones Legítimas (+1)')
axes[0].scatter(X_total[y_real==-1, 0], X_total[y_real==-1, 1], c='crimson', marker='X', s=80, edgecolors='k', label='Fraudes Reales (-1)')
plt.colorbar(cf, ax=axes[0], label='decision_function (>0 Normal, <0 Anomalía)')
axes[0].set_title('Mapa Topológico de Aislamiento y Frontera de Decisión (Nivel 0)', fontsize=11.5, fontweight='bold')
axes[0].legend(loc='upper left', frameon=True)

# Panel 2: Distribución de Scores entre Normales y Fraudes
axes[1].hist(scores_decision[y_real==1], bins=35, color='#1f77b4', alpha=0.75, edgecolor='k', label='Legítimas (Camino Largo -> Score > 0)')
axes[1].hist(scores_decision[y_real==-1], bins=15, color='#d62728', alpha=0.85, edgecolor='k', label='Fraudes (Camino Corto -> Score < 0)')
axes[1].axvline(0.0, color='black', linestyle='--', lw=2.2, label='Umbral de Corte (0.0)')
axes[1].set_title('Separación de Distribuciones del Puntaje de Aislamiento', fontsize=11.5, fontweight='bold')
axes[1].set_xlabel('Puntaje decision_function(X)')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.suptitle('Geometría y Puntajes de Isolation Forest', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 3. Invarianza de Escala: ¿Necesita Isolation Forest Estandarización?

Al igual que los Árboles de Decisión (CART) y Random Forest, **Isolation Forest es completamente invariante a transformaciones lineales monótonas de la escala de las variables**.

¿Por qué? Porque en cada nodo interno se selecciona primero la columna $j$ con probabilidad uniforme $\frac{1}{p}$, y luego el corte $t$ se sortea uniformemente dentro del rango $[\min(x_j), \max(x_j)]$. Multiplicar una columna por $1,000,000$ simplemente multiplica su rango por $1,000,000$, manteniendo intacta la probabilidad geométrica de corte.

Comprobémoslo empíricamente:


In [ ]:
# Prueba de invarianza a la escala
X_desescalado = X_total.copy()
X_desescalado[:, 0] = X_desescalado[:, 0] * 1_000_000 + 50_000  # Escala en millones de USD
X_desescalado[:, 1] = X_desescalado[:, 1] * 0.0001              # Escala milimétrica

if_orig = IsolationForest(n_estimators=200, random_state=42).fit(X_total)
if_des = IsolationForest(n_estimators=200, random_state=42).fit(X_desescalado)

diff_max = np.max(np.abs(if_orig.score_samples(X_total) - if_des.score_samples(X_desescalado)))
print(f"Diferencia máxima en score_samples entre datos normales y datos multiplicados por 1,000,000: {diff_max:.2e}")
print("Conclusión: Isolation Forest es 100% invariante a la escala de las unidades.")


## 4. Conclusiones

1. **Aislamiento Directo sin Distancias**: Isolation Forest no calcula distancias entre pares de puntos ni densidades; mide cuántos cortes rectangulares aleatorios se necesitan para dejar sola a una observación.
2. **Eficiencia por Submuestreo ($\psi = 256$)**: Al entrenar cada árbol sobre una submuestra pequeña de 256 puntos, evita el *swamping* (enmascaramiento por exceso de puntos normales) y opera en tiempo casi constante.
3. **Invarianza de Escala**: No requiere `StandardScaler` ni `RobustScaler`, simplificando drásticamente los pipelines de producción.

En el siguiente cuaderno, **02_Hiperparametros_Contamination_y_Calibracion_Umbrales.ipynb**, estudiaremos cómo ajustar sus hiperparámetros clave (`contamination`, `max_samples`, `max_features`) y cómo calibrar el umbral de alerta en producción.
